In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle    ## to pickle file,ie, deployment; especially when we use StandardScaler, LabelEncoder

In [8]:
## Load the dataset
data=pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [9]:
## Preprocess the data
### Drop irrelevant columns  - since rownumber, custId and surname are irrelevant to find 
# whether the cust will leave the bank or not
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)   #axis=1: col wise- will check col names 
# such as RowNumber etc in col AND axis=0: row wise- will check col names such as RowNumber etc in rows
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


as you can see gender and geography col are categorical, like gender is either male or female.
similarly, geography is either france, spain or germany.

And machine does not understand words so it needs to be converted to numerical form.

Hence we will convert categorical var to numerical, by using 'LabelEncoder' or 'OneHotEncoding'

In [10]:
## Encode categorical variables - Gender
label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [11]:
## Onehot encode 'Geography
#since geography has 3 categories and while giving spain=0,france=1,germany=2 is easy lie did in 
# gender but machine might infer it as if german has high priority than spain and france etc.
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo=OneHotEncoder()
geo_encoder=onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoder

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [12]:
onehot_encoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [13]:
#hence ohe output is as follows
geo_encoded_df=pd.DataFrame(geo_encoder,columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

#if the country is present then 1 or else 0

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [14]:
## Combine one hot encoder columns with the original data
# removing the og geography col and adding the numerical var coln
data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [15]:
## Save the encoders and scaler
# we save encoders and scaler as pickle files so we can reuse the same preprocessing 
# during prediction that we used during training -- this keeps the i/p format consistent

# a scaler learns values like mean and std deviation from training data
# an encoder learns category mappings from training data
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)


In [16]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [17]:
## DiVide the dataset into indepent and dependent features
# Y: Exited is the dependent feature, because it is the value you want the model to predict. 
# X is the set of independent features, because it contains all the input columns used to predict Exited
X=data.drop('Exited',axis=1)
y=data['Exited']

## Split the data in training and tetsing sets
# test_size=0.2 means 20% of the dataset is kept for testing, and the remaining 80% is used for training.
# This is common because the model gets enough data to learn from, 
# while still leaving a separate unseen portion to evaluate performance

# No, random_state does not have to be 42. It can be any integer, and the number itself has 
# no special machine-learning meaning; it just acts as a seed that makes the split reproducible.
# what it means is: you always get the same training and testing rows in the same order. 
# If you change it to random_state=1, you’ll get a different but still reproducible split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

## Scale these features
# You scale X_train using fit_transform() and X_test using transform() 
# X_train goes in fit_transform() because the scaler must learn from training data first.
# X_test goes in transform() because the test data should only be converted using the same training rule

# fit_transform(X_train) =  learn scale from train data, then scale it
# transform(X_test) = use the same learned scale to scale test data

#Tiny example
#If one column has values like 1000, 2000, 3000 and another has 1, 2, 3, scaling makes them 
# comparable so one column does not dominate just because its numbers are bigger
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)


In [18]:
X_train

array([[ 0.35649971,  0.91324755, -0.6557859 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.20389777,  0.91324755,  0.29493847, ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.96147213,  0.91324755, -1.41636539, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.86500853, -1.09499335, -0.08535128, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.15932282,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.47065475,  0.91324755,  1.15059039, ..., -0.99850112,
         1.72572313, -0.57638802]])

In [19]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

In [20]:
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


### ANN Implementation

In [21]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import dateutil

X_train.shape:

x_train.shape shows the dimensions of x_train. If you load a CSV file into x_train as a NumPy array (typically using pandas), then x_train.shape will output a tuple showing (number of rows, number of columns).

x_train.shape = (100, 4)==(rows,col) means 100 samples and 4 features each.

x_train.shape[0] gives the number of samples(rows).

x_train.shape[1] gives the number of features(columns) in 2D data.

In [22]:
X_train.shape(1)

TypeError: 'tuple' object is not callable

In [23]:
X_train.shape[1]

12

In [24]:
(X_train.shape[1],)

(12,)

In order to initiate the 1st hidden layer, we need to give input node count to it as well. Since its connected w i/p layer. The same is not needed for other Hidden layers.

 "input_shape=(X_train.shape[1],)"  fetches the input nodes count from the dataset and connects it w the 1st hidden layer

In [29]:
## Build our ANN model:

model = Sequential([  #initializing sequential netw
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),   ## HL1 Connected wwith input layer
    Dense(32, activation='relu'),      ## HL2
    Dense(1, activation='sigmoid')    ## output layer
])

In [ ]:
model.summary()    #gives summary of node count, total trainable parameters(including weights and biases)

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 64)                832       
                                                                 
 dense_4 (Dense)             (None, 32)                2080      
                                                                 
 dense_5 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [35]:
import tensorflow

# Adam stands for Adaptive Moment Estimation. It's an optimization algorithm that adjusts 
# the model's weights during training to minimize the loss function.
# Adam: Automatically adjusts learning rate for each parameter

#The learning rate is a parameter that determines how large a step the optimizer takes 
# when updating the model's weights during training

#Think of it as the step size when moving towards minimizing the loss function based on gradient
#new_weight = old_weight - learning_rate × gradient
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01)
loss=tensorflow.keras.losses.BinaryCrossentropy()
loss

## Metrics
Metrics measure how well your ANN is performing during and after training. They tell you if the model is learning correctly and making good predictions

## Loss
A loss function measures how wrong your model's predictions are compared to the actual values. It tells the model how badly it's performing so it can learn to improve.

For multi class: use sparse_categorical_crossentropy

In [37]:
## compile the model
model.compile(optimizer=opt, loss=loss, metrics=['accuracy'])  #metrics in list bcz of multiple metric can be given

In [ ]:
## setup the tensorboard
import datetime
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")   #strftime stands for 
#"string format time" and is used to convert a datetime object to a formatted string.
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

- histogram_freq=1 means calculate and log weight histograms every 1 epoch.

- In machine learning, an epoch is one complete pass through the entire training dataset during training. During that single epoch, the neural network sees every training sample exactly once, performing a forward pass (making predictions) and a backward pass (updating weights to reduce error).

- e

the concept behind earlystopping is that:
when we're training the neural network, we can train the model for any no. of epochs.

Say we're training the model using 100 epochs, but after 20 epochs the Loss value is not decreasing so its not useful if we run it for 100 epochs. Hence, here the earlystopping comes in play, we tell earlystopping that if the loss value is not decreasing for next 4-5(Patience parameter) epochs then stop the training there.

In [ ]:
## Set up Early Stopping

#patience - checks for next 10 epochs, if the value does not decrease then stop
# restore_best_weights- Whether to go back to best version
# val_loss stands for validation loss. 
# monitor='val_loss' = Watch how well the model does on new data, and stop when it stops improving.
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [4]:
# Training the model

history=model.fit(
    X_train, y_train, validation_data=(X_test,y_test), epochs=100,
    callbacks=[tensorflow_callback,early_stopping_callback]
)

NameError: name 'model' is not defined

In [44]:
# saves the ENTIRE trained model to a file.

model.save('model.h5')

c:\Users\Srushti\Desktop\SRUSHTI PYTHON\17-ANN Classification\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [2]:
# Load tensorboard extension
# %load_ext tensorboard is a Jupyter Notebook magic command that loads the TensorBoard extension.
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [3]:
%tensorboard --logdir logs/fit/ 
# starts TensorBoard and points it to the folder where your training logs are stored
# Opens TensorBoard visualization in your notebook, showing the training metrics saved in the logs/fit folder.

for some reason, the tensoboard is not visible here in jupyter nb, but in localhost, it shows - so some vs code problem

for localhost link, run "tensorboard --logdir logs/fit" in cmd


for prediction, go to prediction.ipynb